# ACC Pipeline — CARLA Ground-Truth Version

Runs against a CARLA simulator instead of uploaded images. For now this uses CARLA's own
simulation state (road type, off-road, weather, pedestrian/vehicle distance, traffic
lights, speed limits) instead of the trained CV models, via `carla_integration.py`
(keep that file in the same folder as this notebook).

The voice assistant (propose→confirm, ElevenLabs TTS, Whisper transcription) is
unchanged from the original pipeline — only the perception step changed.

**Before running:** start the CARLA server separately (`CarlaUE4.exe` / `./CarlaUE4.sh`),
then run this notebook's cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install mutagen elevenlabs groq openai-whisper sounddevice scipy pillow matplotlib




## 2. Imports

In [ ]:
import os
import re
import json
import time
import numpy as np
from PIL import Image

import carla
import carla_integration as ci

import whisper
from groq import Groq
from elevenlabs.client import ElevenLabs
from elevenlabs.play import save as save_audio
from mutagen.mp3 import MP3

import sounddevice as sd
from scipy.io.wavfile import write as write_wav

from IPython.display import Audio, display

print("Imports OK")


## 3. Config

In [ ]:
CARLA_HOST = "127.0.0.1"
CARLA_PORT = 2000

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
ELEVENLABS_VOICE_ID = "JBFqnCBsd6RMkjVDRZzb"
ELEVENLABS_MODEL_ID = "eleven_multilingual_v2"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LLM_MODEL = "openai/gpt-oss-120b"

ANNOUNCEMENT_LANGUAGE = "fr"


## 4. Connect to CARLA and spawn the ego vehicle + camera

In [ ]:
client, world = ci.connect(host=CARLA_HOST, port=CARLA_PORT)
world_map = world.get_map()
vehicle, camera, image_q = ci.spawn_vehicle_with_camera(world)
print("Connected. Map:", world_map.name)


## 5. Speed logic — base speed + reason-code overrides
Priority, highest to lowest: pedestrian close → stop / red light → crosswalk → speed bump →
posted speed limit → car too close → adverse weather. The first two are automatic (no
confirmation, see `AUTOMATIC_REASON_CODES`); everything after is a confirmable suggestion.
Unchanged from the original notebook.

In [ ]:
BASE_SPEED_BY_ROAD_TYPE = {
    "autoroute": 120,
    "urbaine": 60,
    "rurale": 100,
    "off-road": 50,
}
DAMAGE_SPEED_REDUCTION = 0.30

CROSSWALK_SPEED_CAP = 30
SPEED_BUMP_SPEED_CAP = 30
SAFETY_DISTANCE_SPEED_REDUCTION = 0.20
WEATHER_SPEED_REDUCTION = 0.20
PEDESTRIAN_FAR_SPEED_REDUCTION = 0.30

AUTOMATIC_REASON_CODES = {"pedestrian_stop", "stop_sign", "red_light"}


def get_base_speed(road_type, is_damaged):
    speed = BASE_SPEED_BY_ROAD_TYPE[road_type]
    if is_damaged:
        speed = speed * (1 - DAMAGE_SPEED_REDUCTION)
    return round(speed, 1)


def apply_overrides(
    current_speed,
    detected_signs,
    pedestrian_close=False,
    pedestrian_far=False,
    crosswalk_ahead=False,
    bump_ahead=False,
    car_too_close=False,
    weather_condition="clear",
):
    """
    Returns (final_speed, reason_code, reason_details). reason_code is one of:
    "pedestrian_stop", "stop_sign", "red_light", "crosswalk", "speed_bump", "speed_limit",
    "pedestrian_far", "safety_distance", "weather", or None.
    """
    if pedestrian_close:
        return 0, "pedestrian_stop", {}

    for sign in detected_signs:
        if sign["class"] == "Stop":
            return 0, "stop_sign", {}
        if sign["class"] == "Red Light":
            return 0, "red_light", {}

    final_speed = current_speed
    reason_code = None
    reason_details = {}

    if crosswalk_ahead and CROSSWALK_SPEED_CAP < final_speed:
        final_speed = CROSSWALK_SPEED_CAP
        reason_code, reason_details = "crosswalk", {"speed": CROSSWALK_SPEED_CAP}

    if bump_ahead and SPEED_BUMP_SPEED_CAP < final_speed:
        final_speed = SPEED_BUMP_SPEED_CAP
        reason_code, reason_details = "speed_bump", {"speed": SPEED_BUMP_SPEED_CAP}

    for sign in detected_signs:
        match = re.search(r"Speed Limit (\d+)", sign["class"])
        if match:
            posted_limit = int(match.group(1))
            if posted_limit < final_speed:
                final_speed = posted_limit
                reason_code, reason_details = "speed_limit", {"speed": posted_limit}

    if pedestrian_far:
        final_speed = round(final_speed * (1 - PEDESTRIAN_FAR_SPEED_REDUCTION), 1)
        reason_code, reason_details = "pedestrian_far", {"speed": final_speed}

    if car_too_close:
        final_speed = round(final_speed * (1 - SAFETY_DISTANCE_SPEED_REDUCTION), 1)
        reason_code, reason_details = "safety_distance", {"speed": final_speed}

    if weather_condition == "adverse":
        final_speed = round(final_speed * (1 - WEATHER_SPEED_REDUCTION), 1)
        reason_code, reason_details = "weather", {"speed": final_speed}

    return final_speed, reason_code, reason_details


## 6. Voice assistant — intent understanding, propose→confirm, ElevenLabs TTS
Unchanged from the original notebook. `understand_and_respond()` only ever *proposes* a
command and asks a confirmation question; `dispatch_command()` only ever runs from
`confirm_pending_command()`.

In [ ]:
whisper_model = whisper.load_model("base")

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path)
    return result["text"].strip(), result["language"]


groq_client = Groq(api_key=GROQ_API_KEY)

VEHICLE_TOOL = {
    "type": "function",
    "function": {
        "name": "vehicle_command",
        "description": (
            "Call this when the user gives an instruction that changes something "
            "about the car (window, speed, cruise speed, eco mode, safety distance). "
            "Do NOT call this for questions, greetings, or general conversation."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": [
                        "close_window", "open_window",
                        "set_speed_limit", "set_cruise_speed",
                        "enable_eco_mode",
                        "increase_safety_distance", "decrease_safety_distance",
                        "decrease_speed", "increase_speed",
                    ],
                },
                "value": {
                    "type": "integer",
                    "description": "Speed in km/h. Only used for set_speed_limit and set_cruise_speed.",
                },
            },
            "required": ["action"],
        },
    },
}

CONFIRMATION_TOOL = {
    "type": "function",
    "function": {
        "name": "record_confirmation",
        "description": "Call this to record whether the driver confirmed or declined the pending action.",
        "parameters": {
            "type": "object",
            "properties": {
                "confirmed": {
                    "type": "boolean",
                    "description": "true if the driver agreed/said yes, false if they declined/said no.",
                },
            },
            "required": ["confirmed"],
        },
    },
}

SYSTEM_PROMPT = """You are the voice assistant embedded in an adaptive cruise control (ADAS) system in a car.

Always reply in the exact same language the user just spoke to you in, whatever language that is.

You have two jobs:
1. If the user gives a command to change something about the car (window, speed limit, cruise speed, eco mode, safety distance, speed up/slow down), call the vehicle_command tool with the right action (and value in km/h if relevant). IMPORTANT: nothing is applied yet at this point -- along with the tool call, phrase your spoken reply as a short CONFIRMATION QUESTION asking the driver if they want you to do it (e.g. "Do you want me to set cruise speed to 100?"), in the same language the user spoke. The action only actually happens after the driver confirms on the next turn.
2. For anything else -- greetings, small talk, "who are you", "what do you do", general questions -- just answer normally and conversationally, like any helpful voice assistant would. Keep replies short (1-3 sentences), since this is spoken out loud in a car, not read on a screen.

Only call the tool for real commands. Never invent vehicle behavior you were not asked for.
"""

_pending_command = {"data": None, "question": None}

CONVERSATION_HISTORY_TURNS = 5
_conversation_history = []


def _append_history(user_text, assistant_text):
    _conversation_history.append({"role": "user", "text": user_text})
    _conversation_history.append({"role": "assistant", "text": assistant_text})
    max_entries = CONVERSATION_HISTORY_TURNS * 2
    if len(_conversation_history) > max_entries:
        del _conversation_history[: len(_conversation_history) - max_entries]


def _history_as_messages():
    return [{"role": turn["role"], "content": turn["text"]} for turn in _conversation_history]


def dispatch_command(action, value, vehicle_state):
    if action == "set_speed_limit" and value is not None:
        vehicle_state["speed_limit_kmh"] = value
    elif action == "set_cruise_speed" and value is not None:
        vehicle_state["cruise_speed_kmh"] = value
    elif action == "enable_eco_mode":
        vehicle_state["eco_mode"] = True
    elif action == "increase_safety_distance":
        vehicle_state["safety_distance_level"] += 1
    elif action == "decrease_safety_distance":
        vehicle_state["safety_distance_level"] = max(1, vehicle_state["safety_distance_level"] - 1)
    elif action == "decrease_speed":
        vehicle_state["cruise_speed_kmh"] = max(0, vehicle_state["cruise_speed_kmh"] - 10)
    elif action == "increase_speed":
        vehicle_state["cruise_speed_kmh"] += 10
    elif action == "close_window":
        vehicle_state["window_open"] = False
    elif action == "open_window":
        vehicle_state["window_open"] = True
    return vehicle_state


def _judge_yes_no(confirmation_text, question_asked):
    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    f'You are the voice assistant in a car. You just asked the driver: "{question_asked}"\n'
                    "The driver has now replied. Call record_confirmation with confirmed=true if they "
                    "agreed/said yes, confirmed=false if they declined/said no.\n"
                    "Also write one short spoken sentence in the same language the driver just used: "
                    "if confirmed, say you're doing it now; if declined, acknowledge you won't change anything."
                ),
            },
            {"role": "user", "content": confirmation_text},
        ],
        tools=[CONFIRMATION_TOOL],
        tool_choice="auto",
    )

    message = response.choices[0].message

    confirmed = False
    if message.tool_calls:
        for call in message.tool_calls:
            if call.function.name == "record_confirmation":
                args = json.loads(call.function.arguments)
                confirmed = bool(args.get("confirmed"))

    spoken_reply = (message.content or "").strip()
    if not spoken_reply:
        spoken_reply = "Done." if confirmed else "Okay, no changes made."

    return confirmed, spoken_reply


def understand_and_respond(user_text, vehicle_state):
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + _history_as_messages()
        + [{"role": "user", "content": user_text}]
    )

    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=[VEHICLE_TOOL],
        tool_choice="auto",
    )

    message = response.choices[0].message

    pending_action = None
    if message.tool_calls:
        for call in message.tool_calls:
            if call.function.name == "vehicle_command":
                args = json.loads(call.function.arguments)
                pending_action = {"action": args.get("action"), "value": args.get("value")}

    spoken_text = (message.content or "").strip()
    if not spoken_text:
        spoken_text = "Done." if pending_action is None else "Do you confirm?"

    if pending_action is not None:
        _pending_command["data"] = pending_action
        _pending_command["question"] = spoken_text

    _append_history(user_text, spoken_text)

    return spoken_text, vehicle_state, pending_action is not None


def confirm_pending_command(confirmation_text, vehicle_state):
    pending = _pending_command["data"]
    if pending is None:
        return vehicle_state, "There is no pending command to confirm."

    confirmed, spoken_reply = _judge_yes_no(confirmation_text, _pending_command["question"])
    if confirmed:
        vehicle_state = dispatch_command(pending["action"], pending["value"], vehicle_state)

    _append_history(confirmation_text, spoken_reply)

    _pending_command["data"] = None
    _pending_command["question"] = None
    return vehicle_state, spoken_reply


elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

def speak(text, output_path="response.mp3", voice_id=ELEVENLABS_VOICE_ID, language_code="en"):
    audio = elevenlabs_client.text_to_speech.convert(
        text=text,
        voice_id=voice_id,
        model_id=ELEVENLABS_MODEL_ID,
        language_code=language_code,
        output_format="mp3_44100_128",
    )
    save_audio(audio, output_path)
    return output_path


def get_audio_duration(path, buffer_seconds=0.5):
    try:
        return MP3(path).info.length + buffer_seconds
    except Exception:
        return 3.0


## 7. Record your own voice to give a manual command (optional)
Independent of the camera/CARLA loop below -- run this cell any time to test the
driver-initiated command flow ("open the window", "set cruise speed to 100", etc).

In [ ]:
def record_audio(duration=4, samplerate=16000, output_path="mic_recording.wav"):
    print(f"Recording for {duration} seconds -- speak now, any language...")
    recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='int16')
    sd.wait()
    write_wav(output_path, samplerate, recording)
    print(f"Recording saved to {output_path}")
    return output_path


if "vehicle_state" not in dir():
    vehicle_state = {
        "cruise_speed_kmh": 50,
        "speed_limit_kmh": None,
        "eco_mode": False,
        "safety_distance_level": 2,
        "window_open": True,
    }

audio_path = record_audio(duration=4)
text, detected_language = transcribe_audio(audio_path)
print(f"\nWhisper heard ({detected_language}): {text!r}")

response_text, vehicle_state, needs_confirmation = understand_and_respond(text, vehicle_state)
print(f"Assistant: {response_text}")

response_audio = speak(response_text, language_code=detected_language)
display(Audio(response_audio, autoplay=True))
time.sleep(get_audio_duration(response_audio))

if needs_confirmation:
    print("\nWaiting for your confirmation (say yes/no, in any language)...")
    confirm_audio_path = record_audio(duration=4, output_path="confirmation.wav")
    confirmation_text, confirm_language = transcribe_audio(confirm_audio_path)
    print(f"Whisper heard ({confirm_language}): {confirmation_text!r}")

    vehicle_state, confirm_response = confirm_pending_command(confirmation_text, vehicle_state)
    print(f"Assistant: {confirm_response}")

    confirm_audio = speak(confirm_response, language_code=confirm_language)
    display(Audio(confirm_audio, autoplay=True))

print("\nUpdated vehicle state:", vehicle_state)


## 8. Speed-change announcements (reason-code driven)
`AUTOMATIC_REASON_CODES` (pedestrian_stop, stop_sign, red_light) are announced immediately
and applied immediately, no confirmation asked. Everything else is a suggestion: announced
as a question and only applied once confirmed. `propose_or_announce()` already only speaks
when `final_speed_kmh` actually changes since the last call — it won't repeat itself every
frame.

In [ ]:
SUGGESTION_QUESTION_TEMPLATES = {
    "crosswalk":       "Passage pi\u00e9ton d\u00e9tect\u00e9. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "speed_bump":      "Dos d'\u00e2ne d\u00e9tect\u00e9. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "speed_limit":     "Limite de vitesse affich\u00e9e : {speed} kilom\u00e8tres heure. Voulez-vous l'appliquer ?",
    "pedestrian_far":  "Pi\u00e9ton rep\u00e9r\u00e9 au loin. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "safety_distance": "V\u00e9hicule proche devant. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "weather":         "Conditions m\u00e9t\u00e9o d\u00e9grad\u00e9es. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    None:              "Vitesse sugg\u00e9r\u00e9e : {speed} kilom\u00e8tres heure. Voulez-vous l'appliquer ?",
}

AUTOMATIC_ANNOUNCEMENT_TEMPLATES = {
    "pedestrian_stop": "Pi\u00e9ton d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
    "stop_sign":        "Stop d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
    "red_light":         "Feu rouge d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
}

_last_announced_speed = {"value": None}
_pending_suggestion = {"speed": None, "question": None}


def propose_or_announce(suggested_speed, reason_code=None):
    if suggested_speed == _last_announced_speed["value"]:
        return None, None, False

    _last_announced_speed["value"] = suggested_speed

    if reason_code in AUTOMATIC_REASON_CODES:
        text = AUTOMATIC_ANNOUNCEMENT_TEMPLATES[reason_code]
        audio_path = speak(text, output_path="speed_announcement.mp3", language_code=ANNOUNCEMENT_LANGUAGE)
        return text, audio_path, False

    template = SUGGESTION_QUESTION_TEMPLATES.get(reason_code, SUGGESTION_QUESTION_TEMPLATES[None])
    text = template.format(speed=suggested_speed)
    audio_path = speak(text, output_path="speed_announcement.mp3", language_code=ANNOUNCEMENT_LANGUAGE)

    _pending_suggestion["speed"] = suggested_speed
    _pending_suggestion["question"] = text
    return text, audio_path, True


def confirm_pending_suggestion(confirmation_text, vehicle_state):
    if _pending_suggestion["speed"] is None:
        return vehicle_state, "There is no pending suggestion to confirm."

    confirmed, spoken_reply = _judge_yes_no(confirmation_text, _pending_suggestion["question"])
    if confirmed:
        vehicle_state["cruise_speed_kmh"] = _pending_suggestion["speed"]

    _append_history(confirmation_text, spoken_reply)

    _pending_suggestion["speed"] = None
    _pending_suggestion["question"] = None
    return vehicle_state, spoken_reply


if "vehicle_state" not in dir():
    vehicle_state = {
        "cruise_speed_kmh": 50,
        "speed_limit_kmh": None,
        "eco_mode": False,
        "safety_distance_level": 2,
        "window_open": True,
    }


## 9. Main CARLA loop — ground-truth mode
Each step: advance the sim one tick, pull the camera frame (kept around in case you want
to look at it, not required for ground-truth mode), get road type / pedestrian / crosswalk
/ vehicle-distance / weather / speed-limit / traffic-light straight from CARLA, run it
through your unchanged speed logic, and let the voice assistant announce/confirm changes
exactly as it did with real image classifications.

In [ ]:
N_STEPS = 500

try:
    for step in range(N_STEPS):
        frame = ci.tick_and_get_frame(world, image_q)  # numpy RGB array

        result = ci.build_result_from_ground_truth(
            world, world_map, vehicle,
            get_base_speed=get_base_speed,
            apply_overrides=apply_overrides,
        )

        spoken_text, audio_path, needs_confirmation = propose_or_announce(
            result["final_speed_kmh"], result["reason_code"]
        )

        if spoken_text:
            print(f"[step {step}] Assistant says: {spoken_text}  (reason: {result['reason_code']})")
            display(Audio(audio_path, autoplay=True))
            time.sleep(get_audio_duration(audio_path))

            if needs_confirmation:
                print("Waiting for your confirmation (say yes/no, in any language)...")
                confirm_audio_path = record_audio(duration=4, output_path="suggestion_confirmation.wav")
                confirmation_text, confirm_language = transcribe_audio(confirm_audio_path)
                vehicle_state, confirm_response = confirm_pending_suggestion(confirmation_text, vehicle_state)
                print(f"Assistant says: {confirm_response}")

                confirm_audio = speak(confirm_response, output_path="suggestion_confirm_response.mp3",
                                       language_code=confirm_language)
                display(Audio(confirm_audio, autoplay=True))

finally:
    ci.shutdown(world, vehicle, camera)
    print("CARLA actors cleaned up, synchronous mode disabled.")
